# UTH Computer Vision Coursework — Topic 11
## Notebook 03: Meta DINOv2 Hard-Sample Mining & Closed-Loop Benchmark
- **Nhóm thực hiện**: Nhóm 4
- **Đề tài**: *Stable Diffusion-Based Synthetic Facial Dataset Generation for Deepfake Detection*
- **Mục tiêu**: Phân tích các Mẫu Khó (`Hard Samples`) vượt qua được mạng DINOv2 baseline và kiểm chứng hiệu quả fine-tuning đối chứng $D_0 \to D_1 \to D_2$.


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image


### 1. Phân tích lỗ hổng mô hình gốc qua Mẫu Khó (`data/hard_samples/`)


In [ ]:
audit_file = Path('data/hard_samples/audit_reports/hard_sample_demographics.json')
if audit_file.exists():
    with open(audit_file, 'r', encoding='utf-8') as f:
        audit = json.load(f)
    print('Báo cáo phân tích Mẫu Khó (Hard Samples Audit):')
    print(json.dumps(audit, indent=2))
else:
    print('File audit chưa được tạo. Chạy: python3 -m src.pipelines.mine_hard_samples')


### 2. So sánh kết quả thực nghiệm 3 giai đoạn đối chứng ($D_0 \to D_1 \to D_2$)


In [ ]:
benchmark_log = Path('experiments/logs/benchmark_comparison.json')
if benchmark_log.exists():
    with open(benchmark_log, 'r', encoding='utf-8') as f:
        bench = json.load(f)
    
    stages = ['D0 (Baseline)', 'D1 (Synthetic)', 'D2 (Hard Boosted)']
    losses = [bench['D0_baseline']['loss'], bench['D1_synthetic']['loss'], bench['D2_hard_samples']['loss']]
    aucs = [bench['D0_baseline']['roc_auc'], bench['D1_synthetic']['roc_auc'], bench['D2_hard_samples']['roc_auc']]
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].bar(stages, losses, color=['#95a5a6', '#f39c12', '#27ae60'])
    axes[0].set_title('Test Loss Qua Các Giai Đoạn (Càng thấp càng tốt)')
    axes[0].set_ylabel('Binary Cross Entropy Loss')
    
    axes[1].bar(stages, aucs, color=['#95a5a6', '#f39c12', '#2980b9'])
    axes[1].set_title('ROC-AUC Score')
    axes[1].set_ylim([0.0, 1.1])
    axes[1].set_ylabel('ROC-AUC')
    
    plt.tight_layout()
    plt.show()
    
    print('Mức cải thiện delta:')
    print(json.dumps(bench.get('deltas', {}), indent=2))


### 3. Lưới ảnh đối chứng trực quan toàn diện (Visual Gallery)


In [ ]:
gallery_path = Path('experiments/predictions/real_vs_synthetic_gallery.png')
if gallery_path.exists():
    im = Image.open(gallery_path)
    plt.figure(figsize=(14, 10))
    plt.imshow(im)
    plt.axis('off')
    plt.title('Lưới Ảnh Đối Chứng: Real Faces vs Synthetic SD vs Mined Hard Samples', fontsize=14, fontweight='bold')
    plt.show()
else:
    print('Chưa có file gallery, chạy: python3 -m scripts.build_visual_gallery')
